# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [5]:
import asyncio
import json
import os
import re
import time
from pathlib import Path
from typing import Optional

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [6]:
DATA_DIR = Path('data')   # data files are stored in mp1/data

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [7]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    return [{
        'role': 'user',
        'content': f"""
Extract the following fields from this job posting snippet:
- company
- role
- years_experience_required

Return valid JSON only, with exactly these keys:
{{
  "company": "...",
  "role": "...",
  "years_experience_required": <integer or null>
}}

Rules:
- Use the exact company name as written in the snippet.
- Use the title/role as written in the snippet, preserving wording where possible.
- If the snippet does not state a years requirement, return null.
- Do not guess. Do not add extra keys.
- Strip trailing punctuation and extra whitespace if needed.

Snippet:
{snippet_text}
"""
    }]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    examples = """
Example 1:
Snippet: "We are hiring a Data Analyst at Northwind Ltd. Preferred 2 years of experience."
Output:
{"company": "Northwind Ltd", "role": "Data Analyst", "years_experience_required": 2}

Example 2:
Snippet: "Acme Corp is looking for a Senior Software Engineer with at least 5 years of experience."
Output:
{"company": "Acme Corp", "role": "Senior Software Engineer", "years_experience_required": 5}

Example 3:
Snippet: "Hooli is hiring a Junior Frontend Developer. No specific experience required."
Output:
{"company": "Hooli", "role": "Junior Frontend Developer", "years_experience_required": null}
"""
    return [{
        'role': 'user',
        'content': f"""
Use the examples below to extract the requested fields from the target snippet.

{examples}

Task:
Extract the following fields from the job posting below:
- company
- role
- years_experience_required

Return valid JSON only with exactly these keys:
{{
  "company": "...",
  "role": "...",
  "years_experience_required": <integer or null>
}}

Rules:
- If no years requirement is stated, return null.
- If the snippet says 'around 6 years' or '7+', take the stated number (6 or 7).
- Do not infer missing information.
- Do not add extra keys.

Snippet:
{snippet_text}
"""
    }]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    schema = """
{
  "company": "string",
  "role": "string",
  "years_experience_required": "integer | null"
}
"""
    return [
        {
            'role': 'system',
            'content': """
You are an expert recruiting analyst. Your job is to read a job posting excerpt and extract three fields:
1) company
2) role
3) years_experience_required

Return a valid JSON object that matches this schema exactly:
""" + schema + """

Strict instructions:
- Use only information stated in the snippet.
- If a field is not explicitly stated, return null for that field.
- Never invent years of experience, role details, or company names.
- Keep values concise and faithful to the original wording.
- Output JSON only, without markdown fences or explanatory prose.
"""
        },
        {
            'role': 'user',
            'content': f"""
Snippet:
{snippet_text}
"""
        },
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    return [
        {
            'role': 'user',
            'content': f"""
Extract these three fields from the job posting snippet:
- company
- role
- years_experience_required

Think step by step quietly before answering:
1. Identify the company name.
2. Identify the job title/role.
3. Check whether the snippet states a years requirement; if it does not, set null.
4. Normalize the years value to an integer when possible.
5. Output valid JSON only with exactly these keys:
{{
  "company": "...",
  "role": "...",
  "years_experience_required": <integer or null>
}}

Important:
- Do not infer missing information.
- If the years requirement is not stated, return null.
- Ignore surrounding marketing text that is not part of the required extraction.
- Do not output extra commentary after the JSON.

Snippet:
{snippet_text}
"""
        }
    ]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}


## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [8]:

def parse_response(text: str) -> Optional[dict]:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    if text is None:
        return None

    cleaned = text.strip()
    if cleaned.startswith('```'):
        cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r'\s*```\s*$', '', cleaned, flags=re.IGNORECASE)

    # Try to isolate the JSON object if there is extra prose.
    match = re.search(r'\{.*\}', cleaned, flags=re.DOTALL)
    if match:
        cleaned = match.group(0)

    try:
        parsed = json.loads(cleaned)
        return parsed if isinstance(parsed, dict) else None
    except json.JSONDecodeError:
        return None


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    strategy_fn = STRATEGIES[strategy_name]
    messages = strategy_fn(snippet['snippet'])

    t0 = time.perf_counter()
    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=TEMPERATURE,
    )
    elapsed = time.perf_counter() - t0

    raw_text = response.choices[0].message.content or ''
    parsed = parse_response(raw_text)

    prompt_tokens = getattr(response.usage, 'prompt_tokens', 0) if response.usage else 0
    completion_tokens = getattr(response.usage, 'completion_tokens', 0) if response.usage else 0
    cost_in = prompt_tokens * RATES[MODEL]['in']
    cost_out = completion_tokens * RATES[MODEL]['out']
    cost_usd = cost_in + cost_out

    return {
        'strategy': strategy_name,
        'snippet_id': snippet['id'],
        'raw_response': raw_text,
        'parsed_extraction': parsed,
        'latency_s': round(elapsed, 4),
        'cost_usd': round(cost_usd, 8),
        'prompt_tokens': prompt_tokens,
        'completion_tokens': completion_tokens,
    }


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    tasks = [
        run_one(strategy_name, snippet)
        for strategy_name in STRATEGIES
        for snippet in snippets
    ]
    return await asyncio.gather(*tasks)


In [9]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': '{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5\n}',
 'parsed_extraction': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5},
 'latency_s': 2.0967,
 'cost_usd': 4.35e-05,
 'prompt_tokens': 170,
 'completion_tokens': 30}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [10]:
def score_accuracy(extracted: Optional[dict], gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    if not isinstance(extracted, dict):
        return 0

    def normalise(value, field_name=None):
        if value is None:
            return None
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            value = int(value)
        value = str(value).strip().lower()
        if field_name == 'years_experience_required':
            value = value.replace('+', '').replace('years', '').strip()
        return value

    score = 0
    for field in ['company', 'role', 'years_experience_required']:
        gold_value = normalise(gold.get(field), field)
        extracted_value = normalise(extracted.get(field), field)
        if gold_value == extracted_value:
            score += 1
    return score


async def score_llm_judge(snippet_text: str, extracted: Optional[dict], gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4.

    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    extracted_text = json.dumps(extracted, ensure_ascii=False) if isinstance(extracted, dict) else 'None'
    gold_text = json.dumps(gold, ensure_ascii=False)

    judge_messages = [
        {
            'role': 'system',
            'content': 'You are a strict evaluator for a structured extraction task. Return only an integer from 1 to 4.'
        },
        {
            'role': 'user',
            'content': f"""
Judge the extracted output against the gold answer.

Rubric:
- 4 = all three fields correct
- 3 = two of three correct, no fabricated data
- 2 = one of three correct, or fabricated a field
- 1 = none correct or unparsable

Job snippet:
{snippet_text}

Gold JSON:
{gold_text}

Extracted JSON:
{extracted_text}

Return only a single integer 1, 2, 3, or 4 and nothing else.
"""
        },
    ]

    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=judge_messages,
        temperature=TEMPERATURE,
    )
    text = response.choices[0].message.content or ''
    match = re.search(r'\b([1-4])\b', text)
    return int(match.group(1)) if match else 1


In [11]:
# Apply scoring to all 40 results
scored = []
for result in results:
    snippet = next(s for s in snippets if s['id'] == result['snippet_id'])
    gold = golden[result['snippet_id']]

    result['accuracy'] = score_accuracy(result.get('parsed_extraction'), gold)
    result['parse_success'] = 1 if result.get('parsed_extraction') is not None else 0
    result['llm_judge_score'] = await score_llm_judge(snippet['snippet'], result.get('parsed_extraction'), gold)
    scored.append(result)

print(f'Scored {len(scored)} results.')

Scored 40 results.


## Step 5 — Build the comparison table

In [12]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(3)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.7,1.0,3.7,0.000,1.760
few_shot,2.9,1.0,3.9,0.001,1.886
structured,2.8,1.0,3.9,0.000,2.020
zero_shot,2.7,1.0,3.8,0.000,1.756


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```